# Basic prescription phenotypes

### Spark and Hail setup

In [ ]:
import pyspark
import dxpy
import hail as hl

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

### Environment setup check

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Project: {dxpy.describe(dxpy.PROJECT_CONTEXT_ID)["name"]} ({dxpy.PROJECT_CONTEXT_ID})')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Importing libraries

In [ ]:
import pandas as pd
import json
from pprint import pprint

### Hail datasets configuration

In [ ]:
hl_database_name = 'prescriptions_db'
dispensed_database_name = 'app879030_20250811132217'

input_prescriptions_tb = 'filtered_prescriptions_v6.2.0.ht'
output_phenotypes_tb = 'basic_prescription_phenotypes_v6.2.0.ht'

hl_preffered_partitioning = 24

In [ ]:
hl_database_id = dxpy.find_one_data_object(name=hl_database_name, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']

## Preparing BNF chapter lookup

Creating BNF chapter => substance lookup based on `drugs_final.csv` (drugs of interest by Sylwia).

In [ ]:
import pandas as pd
import json

In [ ]:
drugs_df = pd.read_csv('../../../data/input/drug_list.csv', dtype=str)

Preparing loaded CSV table.

In [ ]:
drugs_df['base_name'] = drugs_df['base_name'].str.split('/')
drugs_df = drugs_df.explode('base_name').reset_index(drop=True)
drugs_df['base_name'] = drugs_df['base_name'].str.lower().str.strip()

In [ ]:
with pd.option_context('display.max_rows', 5):
    display(drugs_df.head(5))

Assigning BNF chapter symbols to substances.

In [ ]:
def drug_class_name(bnf_par_code):
    bnf_par_code = bnf_par_code.strip()
    if bnf_par_code.startswith('02'):
        return 'cardio'
    elif bnf_par_code.startswith('04'):
        return 'cns'
    raise ValueError(f'Invalid BNF Code: "{bnf_par_code}".')

drugs_df['bnf_chapter_sym'] = drugs_df['bnf_paragraph_code'].apply(drug_class_name)

drugs_df = drugs_df[['base_name', 'bnf_chapter_sym']].drop_duplicates()

#### Checking drugs shared by both systems

In [ ]:
drugs_df[drugs_df.duplicated('base_name', keep=False)]

#### Creating final lookup from pandas

In [ ]:
substance_chapter_lkp = drugs_df.groupby('bnf_chapter_sym')['base_name'].agg(set).to_dict()

Checking dictionary integrity.

In [ ]:
all_lkp_substances = set().union(*substance_chapter_lkp.values())
len(all_lkp_substances)

In [ ]:
with open('../../../data/input/substances.json', 'r') as file:
    substances_dict = json.loads(file.read())
len(set(substances_dict.keys()))

In [ ]:
assert all_lkp_substances == set(substances_dict.keys())

## Data loading

### Loading prescriptions

In [ ]:
input_prescriptions_ht = hl.read_table(f'dnax://{hl_database_id}/{input_prescriptions_tb}')
input_prescriptions_ht.count()

### Importing `gp_scripts` data to Hail

In [ ]:
spark.sql(f'USE {dispensed_database_name}')

In [ ]:
%%time
gp_scripts_tb = spark.sql('SELECT DISTINCT eid FROM gp_scripts')
gp_scripts_eids = hl.Table.from_spark(gp_scripts_tb).key_by('eid').cache()
gp_scripts_eids.count()

## Building phenotypes

### Phenotype: number of substances from each drug class and substances exposures

In [ ]:
def hl_dict(items):
    hashmap = {}
    for c in items:
        hashmap[c] = 1
    return hl.literal(hashmap)

Aggregating substances per eid.

In [ ]:
%%time
aggregated_substances = (
    input_prescriptions_ht
    .group_by(input_prescriptions_ht.eid)
    .aggregate(
        unique_substances = hl.agg.explode(lambda s: hl.agg.collect_as_set(s), input_prescriptions_ht.substances))
    .cache()
)

In [ ]:
aggregated_substances.count()

In [ ]:
annotations = {}

for chapter, substances in substance_chapter_lkp.items():
    substances_set = hl_dict(substances)
    annotations[f'{chapter}_substances_num'] = hl.len(aggregated_substances.unique_substances.filter(lambda sub: substances_set.contains(sub)))

all_substances_set = hl_dict(all_lkp_substances)
annotations['unclassified_substances_num'] = hl.len(aggregated_substances.unique_substances.filter(lambda sub: ~(all_substances_set.contains(sub))))

for substance in sorted(all_lkp_substances):
    annotations[f'{substance}__prescribed'] = aggregated_substances.unique_substances.contains(substance)
    
%time substances_phenotypes = aggregated_substances.annotate(**annotations).cache()

In [ ]:
substances_phenotypes.show(5)

In [ ]:
if substances_phenotypes.aggregate(hl.agg.sum(substances_phenotypes.unclassified_substances_num)) > 0:
    raise RuntimeError('Found substance not included in substances class lookup.')

In [ ]:
substances_phenotypes = substances_phenotypes.drop('unique_substances', 'unclassified_substances_num').cache()

### Phenotype: number of prescriptions from each drug class

In [ ]:
annotations = {}

for chapter, substances in substance_chapter_lkp.items():
    substances_set = hl_dict(substances)
    annotations[f'{chapter}_substance_present'] = input_prescriptions_ht.substances.any(lambda sub: substances_set.contains(sub))

%time classfied_prescriptions = input_prescriptions_ht.annotate(**annotations).cache()

In [ ]:
aggregations = {}
for chapter in substance_chapter_lkp.keys():
    aggregations[f'{chapter}_prescriptions_count'] = hl.int(hl.agg.count_where(classfied_prescriptions[f'{chapter}_substance_present']))

%time prescriptions_phenotypes = classfied_prescriptions.group_by(classfied_prescriptions.eid).aggregate(**aggregations).cache()

In [ ]:
for chapter in substance_chapter_lkp.keys():
    phenotype = f'{chapter}_prescriptions_count'
    count_sum = prescriptions_phenotypes.aggregate(hl.agg.sum(prescriptions_phenotypes[phenotype]))
    print(f'{phenotype} sum:\n{count_sum}\n')

## Joining phenotypes in one table

In [ ]:
%time final_phenotypes = prescriptions_phenotypes.join(substances_phenotypes, how = 'outer').cache()

Strange artiacts/bugs checking.

In [ ]:
assert final_phenotypes.count() == prescriptions_phenotypes.count()
assert final_phenotypes.count() == substances_phenotypes.count()

## Adding participants without filtered prescriptions (from `gp_scripts`)

Checking for excessive participants (with filtered prescriptions but without prescriptions in `gp_scripts`):

In [ ]:
excessive_participants_count = final_phenotypes.anti_join(gp_scripts_eids).count()
excessive_participants_count

In [ ]:
# assert excessive_participants_count == 0

Imputing phenotypes for registered participants without filtered prescriptions.

In [ ]:
missed_participants_phenotypes = gp_scripts_eids.anti_join(final_phenotypes).cache()

In [ ]:
annotations = {}
for col_name, col_type in final_phenotypes.row_value.items():
    if col_type.dtype == hl.tint:
        annotations[col_name] = 0
    elif col_type.dtype == hl.tbool:
        annotations[col_name] = False
    else:
        raise ValueError(f'Not supported type "{col_type.dtype}" of phenotype column "{col_name}".')
        
%time missed_participants_phenotypes = missed_participants_phenotypes.annotate(**annotations).cache()

In [ ]:
missed_participants_phenotypes.count()

In [ ]:
%%time
final_phenotypes = (
    final_phenotypes
    .union(missed_participants_phenotypes)
    .repartition(hl_preffered_partitioning)
    .cache()
)

In [ ]:
final_phenotypes.count()

In [ ]:
assert final_phenotypes.count() == (gp_scripts_eids.count() + excessive_participants_count)

## Writing phenotypes table to database

In [ ]:
%time final_phenotypes.write(f'dnax://{hl_database_id}/{output_phenotypes_tb}', overwrite=True)